# LangChain使用之Model I/O
## 1. Model I/O介绍
Model I/O 模块是与语言模型（LLMs）进行交互的 核心组件 ，在整个框架中有着很重要的地位。

所谓的Model I/O，包括输入提示(Format)、调用模型(Predict)、输出解析(Parse)。分别对应着
Prompt Template ， Model 和 Output Parser 。

## 2. Model I/O之调用模型1
### 2.1 LLMs(非对话模型)

In [5]:
from langchain_ollama import OllamaLLM

###########核心代码############
llm = OllamaLLM(model="qwen:7b")
str = llm.invoke("写一首关于春天的诗") # 直接输入字符串
print(str)

春光熹微生绿意，
万物复苏展新姿。
柳条摇曳，如舞者轻盈，
桃花笑靥，似佳人醉人。

风儿温柔，带走了冬日的寒冷，
溪水潺潺，唤醒了大地的生机。

春天啊，你是一首无尽的诗，
在每个角落，都留下了你的足迹。


### 2.2 Chat Models(对话模型)

In [6]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

########核心代码############
chat_model = ChatOllama(model="qwen:7b")
messages = [
SystemMessage(content="我是人工智能助手，我叫小智"),
HumanMessage(content="你好，我是小明，很高兴认识你")
]
response = chat_model.invoke(messages) # 输入消息列表
print(type(response)) # <class 'langchain_core.messages.ai.AIMessage'>
print(response.content)

<class 'langchain_core.messages.ai.AIMessage'>
你好，小明！我也很高兴能和你相识。有什么我能帮助你的吗？


### 2.3 Embedding Model(嵌入模型)
也叫文本嵌入模型，这些模型将 文本 作为输入并返回 浮点数列表 ，也就是Embedding。

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

res1 = embeddings_model.embed_query('我是文档中的数据')
print(res1)

[-0.005971120670437813, 0.1181352436542511, 0.0830983817577362, 0.011212742887437344, 0.02068527415394783, 0.06367464363574982, 0.12943081557750702, -0.01952940970659256, 0.09395983815193176, -0.01008662674576044, 0.07168331742286682, -0.0617380291223526, -0.006280721165239811, -0.0618385374546051, -0.0038902126252651215, -0.00376925733871758, -0.007139479275792837, 0.018814895302057266, -0.08897339552640915, 0.04855788126587868, 0.003366539254784584, 0.0005220387247391045, 0.03224669769406319, 0.03269338235259056, -0.02435752935707569, -0.008826270699501038, -0.06456027179956436, 0.030281925573945045, 0.07276643812656403, -0.016566898673772812, -0.0611230805516243, 0.027421796694397926, 0.029109766706824303, 0.045004263520240784, 0.02511175163090229, -0.011827034875750542, -0.06430739164352417, -0.06335082650184631, -0.0023616475518792868, 0.039809659123420715, 0.017913194373250008, 0.014017906039953232, 0.05884871259331703, -0.05516044422984123, -0.02203972265124321, -0.0031187806744

## 3. Model I/O之调用模型2
### 3.1 关于对话模型的Message(消息)
聊天模型，出了将字符串作为输入外，还可以使用 聊天消息 作为输入，并返回 聊天消息 作为输出。

LangChain有一些内置的消息类型：
- `SystemMessage` ：设定AI行为规则或背景信息。比如设定AI的初始状态、行为模式或对话的总体目标。比如“作为一个代码专家”，或者“返回json格式”。通常作为输入消息序列中的第一个
传递。
- `HumanMessage` ：表示来自用户输入。比如“实现 一个快速排序方法”
- `AIMessage` ：存储AI回复的内容。这可以是文本，也可以是调用工具的请求
- `ChatMessage` ：可以自定义角色的通用消息类型
- `FunctionMessage/ToolMessage` ：函数调用/工具消息，用于函数调用结果的消息类型


In [1]:
# 例子1
from langchain_core.messages import SystemMessage, HumanMessage
messages = [
    SystemMessage(content="我是人工智能助手，我叫小智"),
    HumanMessage(content="你好，我是小明，很高兴认识你")
]
print(messages)

[SystemMessage(content='我是人工智能助手，我叫小智', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，我是小明，很高兴认识你', additional_kwargs={}, response_metadata={})]


In [2]:
# 例子2
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage

messages = [
    SystemMessage(content=["你是一个数学家,只会回答数学问题","每次你都能给出详细的方案"]),
    HumanMessage(content="1 + 2 * 3 = ?"),
    AIMessage(content="1 + 2 * 3 的结果是7")
]
print(messages)

[SystemMessage(content=['你是一个数学家,只会回答数学问题', '每次你都能给出详细的方案'], additional_kwargs={}, response_metadata={}), HumanMessage(content='1 + 2 * 3 = ?', additional_kwargs={}, response_metadata={}), AIMessage(content='1 + 2 * 3 的结果是7', additional_kwargs={}, response_metadata={})]


In [4]:
# 例子3
#1.导入相关包
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
# 2.直接创建不同类型消息
systemMessage = SystemMessage(
content="你是一个AI开发工程师",
additional_kwargs={"tool": "invoke_tool()"}
)
humanMessage = HumanMessage(
content="你能开发哪些AI应用?"
)
aiMessage = AIMessage(
content="我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等"
)
# 3.打印消息列表
messages = [systemMessage,humanMessage,aiMessage]
print(messages)


[SystemMessage(content='你是一个AI开发工程师', additional_kwargs={'你的名字': 'MuziAI'}, response_metadata={}), HumanMessage(content='你能开发哪些AI应用?', additional_kwargs={}, response_metadata={}), AIMessage(content='我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等', additional_kwargs={}, response_metadata={})]


In [5]:
# 例子4
from langchain_core.messages import (
AIMessage,
HumanMessage,
SystemMessage,
ChatMessage
)

# 创建不同类型的消息
system_message = SystemMessage(content="你是一个专业的数据科学家")
human_message = HumanMessage(content="解释一下随机森林算法")
ai_message = AIMessage(content="随机森林是一种集成学习方法...")
custom_message = ChatMessage(role="analyst", content="补充一点关于超参数调优的信息")
print(system_message.content)
print(human_message.content)
print(ai_message.content)
print(custom_message.content)

你是一个专业的数据科学家
解释一下随机森林算法
随机森林是一种集成学习方法...
补充一点关于超参数调优的信息


In [7]:
# 例子5 结合大模型
from langchain_core.messages import SystemMessage,HumanMessage
from langchain_ollama import ChatOllama
chat_model = ChatOllama(
    model="qwen:7b",
)
# 组成消息列表
messages = [
    SystemMessage(content="你是一个擅长人工智能相关学科的专家"),
    HumanMessage(content="请解释一下什么是机器学习？")
]
response = chat_model.invoke(messages)
print(response.content)
print(type(response)) #<class 'langchain_core.messages.ai.AIMessage'>

机器学习（Machine Learning）是一种计算机科学领域，旨在让计算机系统通过经验自动改善性能，而无需显式编程。

在机器学习中，数据被用于训练模型，这些模型可以对新的、未见过的数据进行预测。常见的机器学习算法包括监督学习（如回归和分类）、无监督学习（如聚类）以及强化学习（侧重于环境交互）等。

总之，机器学习是通过分析大量数据来建立数学模型，进而实现自动化任务的一种人工智能技术。
<class 'langchain_core.messages.ai.AIMessage'>


### 3.2 关于多轮对话与上下文记忆

In [8]:
# 获取大模型
from langchain_ollama import ChatOllama

chat_model = ChatOllama(model="qwen:7b")

In [9]:
# 测试1
from langchain_core.messages import SystemMessage, HumanMessage

sys_message = SystemMessage(
    content="我是一个人工智能的助手，我的名字叫小智",
)
human_message = HumanMessage(content="猫王是一只猫吗？")

messages = [sys_message, human_message]
#调用大模型，传入messages
response = chat_model.invoke(messages)
print(response.content)

response1 = chat_model.invoke("你叫什么名字？")
print(response1.content)

不是。"猫王"指的是美国著名的摇滚歌手 Elvis Presley，他的本名是埃尔维斯·普雷斯利。而猫（cat）是一种动物，与这位音乐家无关。
我是来自阿里云的语言模型，我叫通义千问。


In [10]:
# 测试2
from langchain_core.messages import SystemMessage, HumanMessage

sys_message = SystemMessage(
    content="我是一个人工智能的助手，我的名字叫小智",
)
human_message = HumanMessage(content="猫王是一只猫吗？")
human_message1 = HumanMessage(content="你叫什么名字？")

messages = [sys_message, human_message,human_message1]
#调用大模型，传入messages
response = chat_model.invoke(messages)
print(response.content)

我是小智，一个人工智能的助手。关于你的问题，“猫王”通常是指美国流行音乐之王——艾尔维斯·普雷斯利（Elvis Presley），他并不是一只真正的猫。


In [11]:
# 测试3
from langchain_core.messages import SystemMessage, HumanMessage

sys_message = SystemMessage(
    content="我是一个人工智能的助手，我的名字叫小智",
)
human_message = HumanMessage(content="猫王是一只猫吗？")
sys_message1 = SystemMessage(
    content="我可以做很多事情，有需要就找我吧",
)
human_message1 = HumanMessage(content="你叫什么名字？")
messages = [sys_message, human_message,sys_message1,human_message1]
#调用大模型，传入messages
response = chat_model.invoke(messages)
print(response.content)

我是阿里云推出的一种超大规模语言模型，我叫通义千问。


In [12]:
# 测试4
from langchain_core.messages import SystemMessage, HumanMessage
# 第1组
sys_message = SystemMessage(
    content="我是一个人工智能的助手，我的名字叫小智"
)
human_message = HumanMessage(content="猫王是一只猫吗？")
messages = [sys_message, human_message]
# 第2组
sys_message1 = SystemMessage(
content="我可以做很多事情，有需要就找我吧",
)
human_message1 = HumanMessage(content="你叫什么名字？")
messages1 = [sys_message1,human_message1]
#调用大模型，传入messages
response = chat_model.invoke(messages)
print(response.content)
response = chat_model.invoke(messages1)
print(response.content)

"猫王"（The King of Cats）并不是指一只真正的猫。这是一个常见的昵称或尊称，用于赞誉某个人在特定领域具有卓越才能和影响力，就像猫王埃尔维斯·普雷斯利在音乐界的地位一样。
我是通义千问，由阿里云开发。你可以叫我Qwen。有什么问题我可以帮助你解答吗？


In [16]:
# 测试5
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
messages = [
    SystemMessage(content="我是一个人工智能助手，我的名字叫小智"),
    HumanMessage(content="人工智能英文怎么说？"),
    AIMessage(content="AI"),
    HumanMessage(content="你叫什么名字")
]
messages1 = [
    SystemMessage(content="我是一个人工智能助手，我的名字叫小智"),
    HumanMessage(content="很高兴认识你"),
    AIMessage(content="我也很高兴认识你"),
    HumanMessage(content="你叫什么名字")
]
messages2 = [
    SystemMessage(content="我是一个人工智能助手，我的名字叫小智"),
    HumanMessage(content="人工智能英文怎么说？"),
    AIMessage(content="AI"),
    HumanMessage(content="你叫什么名字")
]
chat_model.invoke(messages2)

AIMessage(content='我是小智，一个由阿里云研发的人工智能助手。有什么问题或者需要帮助的吗？', additional_kwargs={}, response_metadata={'model': 'qwen:7b', 'created_at': '2026-01-05T02:14:09.415996Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1149513458, 'load_duration': 58478167, 'prompt_eval_count': 39, 'prompt_eval_duration': 112952916, 'eval_count': 23, 'eval_duration': 850694209, 'logprobs': None, 'model_name': 'qwen:7b', 'model_provider': 'ollama'}, id='lc_run--019b8bee-e689-71e3-9fb8-7cf6d1be7fd8-0', usage_metadata={'input_tokens': 39, 'output_tokens': 23, 'total_tokens': 62})

### 3.3 关于模型调用的方法
#### 3.3.1 流式输出与非流式输出
**非流式输出**

In [1]:
# 例子1
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
#初始化大模型
chat_model = ChatOllama(model="qwen:7b")
# 创建消息
messages = [HumanMessage(content="你好，请介绍一下自己")]
# 非流式调用LLM获取响应
response = chat_model.invoke(messages)
# 打印响应内容
print(response)

content='你好！作为一个人工智能模型，我没有传统意义上的个人身份，但我被设计和编程来帮助解答问题、提供信息和服务。\n\n你可以问我任何你想知道的问题，无论是科技知识、历史事件、还是日常生活中的疑惑，我会尽我所能为你提供帮助。' additional_kwargs={} response_metadata={'model': 'qwen:7b', 'created_at': '2026-01-05T03:29:30.232317Z', 'done': True, 'done_reason': 'stop', 'total_duration': 25761612100, 'load_duration': 5725023700, 'prompt_eval_count': 13, 'prompt_eval_duration': 1740943200, 'eval_count': 56, 'eval_duration': 18209908000, 'logprobs': None, 'model_name': 'qwen:7b', 'model_provider': 'ollama'} id='lc_run--019b8c33-81d1-74f3-8ecb-e7607d8ccc04-0' usage_metadata={'input_tokens': 13, 'output_tokens': 56, 'total_tokens': 69}


In [3]:
# 例子2
# 支持多个消息作为输入
from langchain_core.messages import SystemMessage
messages = [
SystemMessage(content="你是一位乐于助人的助手。你叫于老师"),
HumanMessage(content="你是谁？")
]
response = chat_model.invoke(messages)
print(response.content)

我是于老师，一个虚拟的人工智能助手，专门为大家提供帮助和解答问题。有什么可以帮到你的吗？


**流式输出**

In [6]:
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama

# 初始化大模型
chat_model = ChatOllama(
    model="qwen:7b",
    streaming=True # 启用流式输出
)
# 创建消息
messages = [HumanMessage(content="你好，请介绍一下自己")]
# 流式调用LLM获取响应
print("开始流式输出：")
for chunk in chat_model.stream(messages):
    # 逐个打印内容块
    print(chunk.content, end="", flush=True) # 刷新缓冲区 (无换行符，缓冲区未刷新，内容可能不会立即显示)
print("\n流式输出结束")

开始流式输出：
你好！作为一个人工智能模型，我并没有个人经历可以介绍。但我被设计出来是为了提供帮助和信息的。你可以问我任何问题，我会尽我所能来为你解答。
流式输出结束


**批量调用**

In [8]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
# 初始化大模型
chat_model = ChatOllama(model="qwen:7b")
messages1 = [SystemMessage(content="你是一位乐于助人的智能小助手"),
HumanMessage(content="请帮我介绍一下什么是机器学习"), ]
messages2 = [SystemMessage(content="你是一位乐于助人的智能小助手"),
HumanMessage(content="请帮我介绍一下什么是AIGC"), ]
messages3 = [SystemMessage(content="你是一位乐于助人的智能小助手"),
HumanMessage(content="请帮我介绍一下什么是大模型技术"), ]
messages = [messages1, messages2, messages3]
# 调用batch
response = chat_model.batch(messages)
print(response)

[AIMessage(content='机器学习（Machine Learning，ML）是一门研究计算机如何自动地获取知识并应用到新问题解决中的学科。简单来说，它是指通过让计算机从数据中“学习”模式和规律，而不需要明确的编程规则。\n\n机器学习可以分为监督学习、无监督学习、半监督学习等不同类型。在实际应用中，如自然语言处理（NLP）、图像识别、推荐系统等领域都有大量的机器学习应用。', additional_kwargs={}, response_metadata={'model': 'qwen:7b', 'created_at': '2026-01-05T03:42:42.8309294Z', 'done': True, 'done_reason': 'stop', 'total_duration': 31117442900, 'load_duration': 117861900, 'prompt_eval_count': 27, 'prompt_eval_duration': 4193473600, 'eval_count': 94, 'eval_duration': 26669728200, 'logprobs': None, 'model_name': 'qwen:7b', 'model_provider': 'ollama'}, id='lc_run--019b8c3f-84fd-78c1-9746-99f81b00345e-0', usage_metadata={'input_tokens': 27, 'output_tokens': 94, 'total_tokens': 121}), AIMessage(content='AIGC，全称为Artificial Intelligence General Competition，中文可以理解为“人工智能全能竞赛”或者“人工智能大挑战”。\n\n这个名称通常与举办此类比赛的组织或平台相关。AIGC旨在通过各类AI任务的竞赛，推动人工智能技术的发展，培养更多的AI人才。', additional_kwargs={}, response_metadata={'model': 'qwen:7b', 'created_at': '2026-01-05T03:43:30.2706812Z', 'done': Tr

#### 3.3.3 同步调用与异步调用(了解)
**同步调用**

In [10]:
import time
def call_model():
    # 模拟同步API调用
    print("开始调用模型...")
    time.sleep(5) # 模拟调用等待,单位：秒
    print("模型调用完成。")
def perform_other_tasks():
    # 模拟执行其他任务
    for i in range(5):
        print(f"执行其他任务 {i + 1}")
        time.sleep(1) # 单位：秒
def main():
    start_time = time.time()
    call_model()
    perform_other_tasks()
    end_time = time.time()
    total_time = end_time - start_time
    return f"总共耗时：{total_time}秒"
# 运行同步任务并打印完成时间
main_time = main()
print(main_time)

开始调用模型...
模型调用完成。
执行其他任务 1
执行其他任务 2
执行其他任务 3
执行其他任务 4
执行其他任务 5
总共耗时：10.050298690795898秒


**异步调用**

In [11]:
import asyncio
import time
async def async_call(llm):
    await asyncio.sleep(5) # 模拟异步操作
    print("异步调用完成")
async def perform_other_tasks():
    await asyncio.sleep(5) # 模拟异步操作
    print("其他任务完成")
async def run_async_tasks():
    start_time = time.time()
    await asyncio.gather(
        async_call(None), # 示例调用，使用None模拟LLM对象
        perform_other_tasks()
    )
    end_time = time.time()
    return f"总共耗时：{end_time - start_time}秒"
# # 正确运行异步任务的方式
# if __name__ == "__main__":
# # 使用 asyncio.run() 来启动异步程序
# result = asyncio.run(run_async_tasks())
# print(result)
# 在 Jupyter 单元格中直接调用
result = await run_async_tasks()
print(result)

异步调用完成
其他任务完成
总共耗时：5.012909412384033秒


**异步调用之ainvoke()**

In [12]:
import inspect
print("ainvoke 是协程函数:", inspect.iscoroutinefunction(chat_model.ainvoke))
print("invoke 是协程函数:", inspect.iscoroutinefunction(chat_model.invoke))

ainvoke 是协程函数: True
invoke 是协程函数: False


In [ ]:
import asyncio
import time
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
# 初始化大模型
chat_model = ChatOllama(model="qwen:7b")
# 同步调用（对比组）
def sync_test():
    messages1 = [SystemMessage(content="你是一位乐于助人的智能小助手"),
    HumanMessage(content="请帮我介绍一下什么是机器学习"), ]
    start_time = time.time()
    response = chat_model.invoke(messages1) # 同步调用
    duration = time.time() - start_time
    print(f"同步调用耗时：{duration:.2f}秒")
    return response, duration
# 异步调用（实验组）
async def async_test():
    messages1 = [SystemMessage(content="你是一位乐于助人的智能小助手"),
    HumanMessage(content="请帮我介绍一下什么是机器学习"), ]
    start_time = time.time()
    response = await chat_model.ainvoke(messages1) # 异步调用
    duration = time.time() - start_time
    print(f"异步调用耗时：{duration:.2f}秒")
    return response, duration
# 运行测试
if __name__ == "__main__":
    # 运行同步测试
    sync_response, sync_duration = sync_test()
    print(f"同步响应内容: {sync_response.content[:100]}...\n")
    # 运行异步测试
    async_response, async_duration = asyncio.run(async_test())
    print(f"异步响应内容: {async_response.content[:100]}...\n")
    # 并发测试 - 修复版本
    print("\n=== 并发测试 ===")
    start_time = time.time()
    async def run_concurrent_tests():
        # 创建3个异步任务
        tasks = [async_test() for _ in range(3)]
        # 并发执行所有任务
        return await asyncio.gather(*tasks)
    # 执行并发测试
    results = asyncio.run(run_concurrent_tests())
    total_time = time.time() - start_time
    print(f"\n3个并发异步调用总耗时: {total_time:.2f}秒")
    print(f"平均每个调用耗时: {total_time / 3:.2f}秒")

## 4. Model I/O之Prompt Template
Prompt Template，通过模板管理大模型的输入。
几种不同类型的提示模板：
- `PromptTemplate` ：LLM提示模板，用于生成字符串提示。它使用 Python 的字符串来模板提示。
- `ChatPromptTemplate` ：聊天提示模板，用于组合各种角色的消息模板，传入聊天模型。
- `XxxMessagePromptTemplate` ：消息模板词模板，包括：SystemMessagePromptTemplate、HumanMessagePromptTemplate、AIMessagePromptTemplate、ChatMessagePromptTemplate等
- `FewShotPromptTemplate` ：样本提示词模板，通过示例来教模型如何回答
- `PipelinePrompt` ：管道提示词模板，用于把几个提示词组合在一起使用。
- 自定义模板 ：允许基于其它模板类来定制自己的提示词模板。

**模板导入**

In [15]:
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import FewShotPromptTemplate
from langchain_core.prompts import (
    ChatMessagePromptTemplate,
    SystemMessagePromptTemplate,
    AIMessagePromptTemplate,
    HumanMessagePromptTemplate,
)

### 4.3 具体使用：PromptTemplate
PromptTemplate类，用于快速构建 包含变量 的提示词模板，并通过 传入不同的参数值 生成自定义的提示词。

主要参数介绍：
- template：定义提示词模板的字符串，其中包含 文本 和 变量占位符（如{name}） ；
- input_variables： 列表，指定了模板中使用的变量名称，在调用模板时被替换；
- partial_variables：字典，用于定义模板中一些固定的变量名。这些值不需要再每次调用时被替换。

函数介绍：

format()：给input_variables变量赋值，并返回提示词。利用format() 进行格式化时就一定要赋
值，否则会报错。当在template中未设置input_variables，则会自动忽略。
#### 4.3.2 两种实例化方式
**方式1：使用构造方法**

In [1]:
# 例子1
from langchain_core.prompts import PromptTemplate
# 定义模板：描述主题的应用
template = PromptTemplate(template="请简要描述{topic}的应用。", input_variables=["topic"])
print(template)

input_variables=['topic'] input_types={} partial_variables={} template='请简要描述{topic}的应用。'


In [2]:
# 使用模板生成提示词
prompt_1 = template.format(topic="机器学习")
prompt_2 = template.format(topic="自然语言处理")
print("提示词1:", prompt_1)
print("提示词2:", prompt_2)

提示词1: 请简要描述机器学习的应用。
提示词2: 请简要描述自然语言处理的应用。


In [3]:
# 例子2：定义多变量模板
from langchain_core.prompts import PromptTemplate
#定义多变量模板
template = PromptTemplate(
    template="请评价{product}的优缺点，包括{aspect1}和{aspect2}。",
    input_variables=["product", "aspect1", "aspect2"]
)
#使用模板生成提示词
prompt_1 = template.format(product="智能手机", aspect1="电池续航", aspect2="拍照质量")
prompt_2 = template.format(product="笔记本电脑", aspect1="处理速度", aspect2="便携性")
print("提示词1:", prompt_1)
print("提示词2:", prompt_2)

提示词1: 请评价智能手机的优缺点，包括电池续航和拍照质量。
提示词2: 请评价笔记本电脑的优缺点，包括处理速度和便携性。


**方式2：调用from_template()**

In [4]:
# 例子1
from langchain_core.prompts import PromptTemplate
prompt_template = PromptTemplate.from_template("请给我一个关于{topic}的{type}解释。")
#传入模板中的变量名
prompt = prompt_template.format(type="详细", topic="量子力学")
print(prompt)

请给我一个关于量子力学的详细解释。


In [5]:
# 例子2
#1.导入相关的包
from langchain_core.prompts import PromptTemplate
# 2.定义提示词模版对象
text = """Tell me a joke"""
prompt_template = PromptTemplate.from_template(text)
# 3.默认使用f-string进行格式化（返回格式好的字符串）
prompt = prompt_template.format()
print(prompt)

Tell me a joke


#### 4.3.3 两种新的结构形式
**形式1：部分提示词模版**
在生成prompt前就已经提前初始化部分的提示词，实际进一步导入模版的时候只导入除已初始化的变量即可。

In [6]:
# 方式1：实例化过程中使用partial_variables变量
from langchain_core.prompts import PromptTemplate
#方式2：
template2 = PromptTemplate(
    template="{foo}{bar}",
    input_variables=["foo","bar"],
    partial_variables={"foo": "hello"}
)
prompt2 = template2.format(bar="world")
print(prompt2)

helloworld


In [7]:
# 使用 PromptTemplate.partial() 方法创建部分提示模板
from langchain_core.prompts import PromptTemplate
template1 = PromptTemplate(
    template="{foo}{bar}",
    input_variables=["foo", "bar"]
)
#方式1：
partial_template1 = template1.partial(foo="hello")
prompt1 = partial_template1.format(bar="world")
print(prompt1)

helloworld


In [13]:
from langchain_core.prompts import PromptTemplate
# 完整模板
full_template = """你是一个{role}，请用{style}风格回答：
问题：{question}
答案："""
# 预填充角色和风格
partial_template = PromptTemplate.from_template(full_template).partial(
    role="资深厨师",
    style="专业但幽默"
)
# 只需提供剩余变量
print(partial_template.format(question="如何煎牛排？"))

你是一个资深厨师，请用专业但幽默风格回答：
问题：如何煎牛排？
答案：


In [14]:
prompt_template = PromptTemplate.from_template(
    template = "请评价{product}的优缺点，包括{aspect1}和{aspect2}。",
    partial_variables= {"aspect1":"电池","aspect2":"屏幕"}
)
prompt= prompt_template.format(product="笔记本电脑")
print(prompt)

请评价笔记本电脑的优缺点，包括电池和屏幕。


**形式2：组合提示词(了解)**

In [15]:
from langchain_core.prompts import PromptTemplate
template = (
PromptTemplate.from_template("Tell me a joke about {topic}")
+ ", make it funny"
+ "\n\nand in {language}"
)
prompt = template.format(topic="sports", language="spanish")
print(prompt)

Tell me a joke about sports, make it funny

and in spanish


#### 4.3.4 format() 与 invoke()
只要对象是RunnableSerializable接口类型，都可以使用invoke()，替换前面使用format()的调用方式。

format()，返回值为字符串类型；invoke()，返回值为PromptValue类型，接着调用to_string()返回字符串。

In [16]:
# 例子1
#1.导入相关的包
from langchain_core.prompts import PromptTemplate
# 2.定义提示词模版对象
prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} joke about {content}."
)
# 3.默认使用f-string进行格式化（返回格式好的字符串）
prompt_template.invoke({"adjective":"funny", "content":"chickens"})

StringPromptValue(text='Tell me a funny joke about chickens.')

In [17]:
#1.导入相关的包
from langchain_core.prompts import PromptTemplate
# 2.使用初始化器进行实例化
prompt = PromptTemplate(
input_variables=["adjective", "content"],
template="Tell me a {adjective} joke about {content}")
# 3. PromptTemplate底层是RunnableSerializable接口 所以可以直接使用invoke()调用
prompt.invoke({"adjective": "funny", "content": "chickens"})

StringPromptValue(text='Tell me a funny joke about chickens')

In [18]:
from langchain_core.prompts import PromptTemplate
prompt_template = (
PromptTemplate.from_template("Tell me a joke about {topic}")
+ ", make it funny"
+ " and in {language}"
)
prompt = prompt_template.invoke({"topic":"sports", "language":"spanish"})
print(prompt)

text='Tell me a joke about sports, make it funny and in spanish'


#### 4.3.5 结合LLM调用

In [25]:
from langchain_ollama import OllamaLLM
from langchain_ollama import ChatOllama
# llm = OllamaLLM(model="qwen:7b")
llm = ChatOllama(model="qwen:7b")
prompt_template = PromptTemplate.from_template(
    template="请评价{product}的优缺点，包括{aspect1}和{aspect2}。",
)

# prompt = prompt_template.format(product="笔记本电脑", aspect1="性能", aspect2="电池")
prompt = prompt_template.invoke({"product":"笔记本电脑", "aspect1":"性能", "aspect2":"电池"})
print(type(prompt))
print(prompt)

llm.invoke(prompt)

<class 'langchain_core.prompt_values.StringPromptValue'>
text='请评价笔记本电脑的优缺点，包括性能和电池。'


AIMessage(content='优点：\n\n1. 性能：笔记本电脑具有高度可定制性，可以根据用户的需求选择不同的处理器（如Intel Core i5或i7，AMD Ryzen等），内存大小，显卡类型等，从而提供强大的计算能力和图形处理能力。\n\n2. 便携性：与台式机相比，笔记本电脑体积小，重量轻，方便携带。无论是出差、旅行还是商务会议，都能轻松应对。\n\n3. 学习和工作环境：许多学校和企业都配备有笔记本电脑，为学习者或员工提供必备的工作工具。\n\n缺点：\n\n1. 续航能力：虽然笔记本电脑在便携性上有优势，但它们通常不如台式机电源稳定，因此续航能力相对较弱。对于需要长时间使用设备的用户来说，电池寿命是一个重要的考虑因素。\n\n2. 硬件升级难度：与台式机相比，笔记本电脑的硬件更换和升级往往较为复杂。如想更换显卡或增加内存容量，通常需要拆解笔记本电脑，这在一定程度上增加了维护成本。\n\n3. 价格差异较大：由于性能、品牌、配置等因素的不同，笔记本电脑的价格差异较大。对于预算有限的用户来说，在满足基本需求的前提下选择性价比高的产品更为重要。', additional_kwargs={}, response_metadata={'model': 'qwen:7b', 'created_at': '2026-01-06T07:29:31.5999011Z', 'done': True, 'done_reason': 'stop', 'total_duration': 62746906400, 'load_duration': 65089700, 'prompt_eval_count': 21, 'prompt_eval_duration': 327548900, 'eval_count': 261, 'eval_duration': 62062951800, 'logprobs': None, 'model_name': 'qwen:7b', 'model_provider': 'ollama'}, id='lc_run--019b9235-0cb0-7242-8ab0-3d70c074ad7e-0', usage_metadata={'input_tokens': 21, 'output_tokens': 261, 'total_tokens': 

### 4.4 具体使用：ChatPromptTemplate
ChatPromptTemplate是创建 聊天消息列表 的提示模板。它比普通 PromptTemplate 更适合处理多角色、多轮次的对话场景。

特点：

支持 System / Human / AI 等不同角色的消息模板

对话历史维护

参数类型：列表参数格式是tuple类型（ role :str content :str 组合最常用）

元组的格式为：

(role: str | type, content: str | list[dict] | list[object])
其中 role 是：字符串（如 "system" 、 "human" 、 "ai" ）
#### 4.4.2 两种实例化方式
**方式1：使用构造方法**

In [26]:
from langchain_core.prompts import ChatPromptTemplate
#参数类型这里使用的是tuple构成的list
prompt_template = ChatPromptTemplate([
    # 字符串 role + 字符串 content
    ("system", "你是一个AI开发工程师. 你的名字是 {name}."),
    ("human", "你能开发哪些AI应用?"),
    ("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
    ("human", "{user_input}")
])
#调用format()方法，返回字符串
prompt = prompt_template.invoke(input={"name":"小谷AI","user_input":"你能帮我做什么?"})
print(type(prompt))
print(prompt)

<class 'langchain_core.prompt_values.ChatPromptValue'>
messages=[SystemMessage(content='你是一个AI开发工程师. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能开发哪些AI应用?', additional_kwargs={}, response_metadata={}), AIMessage(content='我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})]


**方式2：调用from_messages()**

In [27]:
# 导入相关依赖
from langchain_core.prompts import ChatPromptTemplate
# 定义聊天提示词模版
chat_template = ChatPromptTemplate.from_messages(
[
("system", "你是一个有帮助的AI机器人，你的名字是{name}。"),
("human", "你好，最近怎么样？"),
("ai", "我很好，谢谢！"),
("human", "{user_input}"),
]
)
# 格式化聊天提示词模版中的变量
messages = chat_template.invoke(input={"name":"小明", "user_input":"你叫什么名字？"})
# 打印格式化后的聊天提示词模版内容
print(messages)

messages=[SystemMessage(content='你是一个有帮助的AI机器人，你的名字是小明。', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，最近怎么样？', additional_kwargs={}, response_metadata={}), AIMessage(content='我很好，谢谢！', additional_kwargs={}, response_metadata={}), HumanMessage(content='你叫什么名字？', additional_kwargs={}, response_metadata={})]


#### 4.4.3 模板调用的几种方式

In [28]:
# invoke()
from langchain_core.prompts import ChatPromptTemplate
#参数类型这里使用的是tuple构成的list
prompt_template = ChatPromptTemplate([
# 字符串 role + 字符串 content
("system", "你是一个AI开发工程师. 你的名字是 {name}."),
("human", "你能开发哪些AI应用?"),
("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
("human", "{user_input}")
])
prompt = prompt_template.invoke({"name":"小谷AI", "user_input":"你能帮我做什么?"})
print(type(prompt))
print(prompt)
print(len(prompt.messages))

<class 'langchain_core.prompt_values.ChatPromptValue'>
messages=[SystemMessage(content='你是一个AI开发工程师. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能开发哪些AI应用?', additional_kwargs={}, response_metadata={}), AIMessage(content='我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})]
4


In [29]:
# format()
from langchain_core.prompts import ChatPromptTemplate
#参数类型这里使用的是tuple构成的list
prompt_template = ChatPromptTemplate([
# 字符串 role + 字符串 content
("system", "你是一个AI开发工程师. 你的名字是 {name}."),
("human", "你能开发哪些AI应用?"),
("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
("human", "{user_input}")
])
#方式1：调用format()方法，返回字符串
prompt = prompt_template.format(name="小谷AI", user_input="你能帮我做什么?")
print(type(prompt))
print(prompt)

<class 'str'>
System: 你是一个AI开发工程师. 你的名字是 小谷AI.
Human: 你能开发哪些AI应用?
AI: 我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.
Human: 你能帮我做什么?


In [30]:
# format_messages()
from langchain_core.prompts import ChatPromptTemplate
prompt_template = ChatPromptTemplate([
("system", "你是一个AI开发工程师. 你的名字是 {name}."),
("human", "你能开发哪些AI应用?"),
("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
("human", "{user_input}")
])
#调用format_messages()方法，返回消息列表
prompt2 = prompt_template.format_messages(name="小谷AI", user_input="你能帮我做什么?")
print(type(prompt2))
print(prompt2)

<class 'list'>
[SystemMessage(content='你是一个AI开发工程师. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能开发哪些AI应用?', additional_kwargs={}, response_metadata={}), AIMessage(content='我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})]


In [31]:
# format_prompt()
from langchain_core.prompts import ChatPromptTemplate
#参数类型这里使用的是tuple构成的list
prompt_template = ChatPromptTemplate([
# 字符串 role + 字符串 content
("system", "你是一个AI开发工程师. 你的名字是 {name}."),
("human", "你能开发哪些AI应用?"),
("ai", "我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等."),
("human", "{user_input}")
])
prompt = prompt_template.format_prompt(name="小谷AI", user_input="你能帮我做什么?")
print(prompt.to_messages())
print(type(prompt.to_messages()))

[SystemMessage(content='你是一个AI开发工程师. 你的名字是 小谷AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能开发哪些AI应用?', additional_kwargs={}, response_metadata={}), AIMessage(content='我能开发很多AI应用, 比如聊天机器人, 图像识别, 自然语言处理等.', additional_kwargs={}, response_metadata={}), HumanMessage(content='你能帮我做什么?', additional_kwargs={}, response_metadata={})]
<class 'list'>


#### 4.4.4 结合LLM

In [32]:
from langchain_core.prompts.chat import ChatPromptTemplate
######1、提供提示词#########
chat_prompt = ChatPromptTemplate.from_messages([
("system", "你是一个数学家，你可以计算任何算式"),
("human", "我的问题：{question}"),
])
# 输入提示
messages = chat_prompt.format_messages(question="我今年18岁，我的舅舅今年38岁，我的爷爷今年72岁，我和舅舅一共多少岁了？")
#print(messages)
######2、提供大模型#########
from langchain_ollama import ChatOllama
chat_model = ChatOllama(model="qwen:7b")
######3、结合提示词，调用大模型#########
# 得到模型的输出
output = chat_model.invoke(messages)
# 打印输出内容
print(output.content)

要计算你和舅舅一共多少岁，你需要将你的年龄加上你舅舅的年龄。

你的年龄 = 18岁
你舅舅的年龄 = 38岁

总年龄 = 你自己的年龄 + 你舅舅的年龄
总年龄 = 18 + 38
总年龄 = 56岁

所以，你和舅舅一共56岁。


#### 4.4.6 插入消息列表：MessagesPlaceholder
多轮对话系统存储历史消息以及Agent的中间步骤处理此功能非常有用。

In [34]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage
prompt_template = ChatPromptTemplate.from_messages([
("system", "You are a helpful assistant"),
MessagesPlaceholder("msgs")
])
prompt_template.invoke({"msgs": [HumanMessage(content="hi!")]})
# prompt_template.format_messages(msgs=[HumanMessage(content="hi!")])

ChatPromptValue(messages=[SystemMessage(content='You are a helpful assistant', additional_kwargs={}, response_metadata={}), HumanMessage(content='hi!', additional_kwargs={}, response_metadata={})])

这将生成两条消息，第一条是系统消息，第二条是我们传入的 HumanMessage。 如果我们传入了 5 条消息，那么总共会生成 6 条消息（系统消息加上传入的 5 条消息）。 这对于将一系列消息插入到特定位置非常有用。

In [35]:
# 举例2：存储对话历史内容
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage
prompt = ChatPromptTemplate.from_messages(
[
("system", "You are a helpful assistant."),
MessagesPlaceholder("history"),
("human", "{question}")
]
)
prompt.format_messages(
history=[HumanMessage(content="1+2*3 = ?"),AIMessage(content="1+2*3=7")],
question="我刚才问题是什么？")

[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='1+2*3 = ?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='1+2*3=7', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='我刚才问题是什么？', additional_kwargs={}, response_metadata={})]

In [39]:
#1.导入相关包
from langchain_core.prompts import (SystemMessagePromptTemplate, ChatPromptTemplate, HumanMessagePromptTemplate, MessagesPlaceholder)
from langchain_core.messages import SystemMessage
# 2.定义消息模板
prompt = ChatPromptTemplate.from_messages([SystemMessagePromptTemplate.from_template("你是{role}"),
MessagesPlaceholder(variable_name="intermediate_steps"),
HumanMessagePromptTemplate.from_template("{query}")
])
# 3.定义消息对象（运行时填充中间步骤的结果）
intermediate = [
SystemMessage(name="search", content="北京: 晴, 25℃")
]
# 4.格式化聊天消息提示词模版
prompt.format_messages(
role="天气预报员",
intermediate_steps=intermediate,
query="北京天气怎么样？"
)

[SystemMessage(content='你是天气预报员', additional_kwargs={}, response_metadata={}),
 SystemMessage(content='北京: 晴, 25℃', additional_kwargs={}, response_metadata={}, name='search'),
 HumanMessage(content='北京天气怎么样？', additional_kwargs={}, response_metadata={})]

### 4.5 具体使用：少量样本示例的提示词模板
在构建prompt时，可以通过构建一个 少量示例列表 去进一步格式化prompt，这是一种简单但强大的指
导生成的方式，在某些情况下可以 显著提高模型性能 。

少量示例提示模板可以由 一组示例 或一个负责从定义的集合中选择 一部分示例 的示例选择器构建。

前者：使用 FewShotPromptTemplate 或 FewShotChatMessagePromptTemplate

后者：使用 Example selectors(示例选择器)

每个示例的结构都是一个 字典 ，其中 键 是输入变量， 值 是输入变量的值。
#### 4.5.2 FewShotPromptTemplate的使用


In [40]:
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts.few_shot import FewShotPromptTemplate
#1、创建示例集合
examples = [
{"input": "北京天气怎么样", "output": "北京市"},
{"input": "南京下雨吗", "output": "南京市"},
{"input": "武汉热吗", "output": "武汉市"}
]
#2、创建PromptTemplate实例
example_prompt = PromptTemplate.from_template(
template="Input: {input}\nOutput: {output}"
)
#3、创建FewShotPromptTemplate实例
prompt = FewShotPromptTemplate(
examples=examples,
example_prompt=example_prompt,
suffix="Input: {input}\nOutput:", # 要放在示例后面的提示模板字符串。
input_variables=["input"] # 传入的变量
)
#4、调用
prompt = prompt.invoke({"input":"长沙多少度"})
print("===Prompt===")
print(prompt)

===Prompt===
text='Input: 北京天气怎么样\nOutput: 北京市\n\nInput: 南京下雨吗\nOutput: 南京市\n\nInput: 武汉热吗\nOutput: 武汉市\n\nInput: 长沙多少度\nOutput:'


In [41]:
from langchain_ollama import ChatOllama
#获取大模型
chat_model = ChatOllama(model="qwen:7b")
#调用
print("===Response===")
response = chat_model.invoke(prompt)
print(response.content) # ???? 结果并非长沙市，是因为7b？

===Response===
长沙当前温度数据，请提供最新的气象信息，以便我能给出准确的回答。


In [42]:
#1、创建提示模板
from langchain_core.prompts import PromptTemplate
# 创建提示模板，配置一个提示模板，将一个示例格式化为字符串
prompt_template = "你是一个数学专家,算式： {input} 值： {output} 使用： {description} "
# 这是一个提示模板，用于设置每个示例的格式
prompt_sample = PromptTemplate.from_template(prompt_template)
#2、提供示例
examples = [
    {"input": "2+2", "output": "4", "description": "加法运算"},
    {"input": "5-2", "output": "3", "description": "减法运算"},
]
#3、创建一个FewShotPromptTemplate对象
from langchain_core.prompts.few_shot import FewShotPromptTemplate
prompt = FewShotPromptTemplate(
examples=examples,
example_prompt=prompt_sample,
suffix="你是一个数学专家,算式: {input} 值: {output}",
input_variables=["input", "output"]
)
print(prompt.invoke({"input":"2*5", "output":"10"}))
#4、初始化大模型，然后调用
from langchain_ollama import ChatOllama
chat_model = ChatOllama(model="qwen:7b")
result = chat_model.invoke(prompt.invoke({"input":"2*5", "output":"10"}))
print(result.content) # 使用: 乘法运算

text='你是一个数学专家,算式： 2+2 值： 4 使用： 加法运算 \n\n你是一个数学专家,算式： 5-2 值： 3 使用： 减法运算 \n\n你是一个数学专家,算式: 2*5 值: 10'
你是一个数学专家，以下是一些具体的解释和使用场景：

1. 算式： 2+2
   值： 4
   使用： 这是一个基础的加法运算。结果表明，将两个2相加得到4。

2. 算式： 5-2
   值： 3
   使用： 这是一个减法运算。通过从5中减去2，我们得到的结果是3。

3. 算式： 2*5
   值： 10
   使用： 这是一个乘法运算。将数字2与5相乘，得到的积是10。


#### 4.5.3 FewShotChatMessagePromptTemplate的使用
除了FewShotPromptTemplate之外，FewShotChatMessagePromptTemplate是专门为 聊天对话场景 设计的少样本（few-shot）提示模板，它继承自 FewShotPromptTemplate ，但针对聊天消息的格式进行了优化。

In [43]:
from langchain_core.prompts import (FewShotChatMessagePromptTemplate, ChatPromptTemplate)
# 1.示例消息格式
examples = [
{"input": "1+1等于几？", "output": "1+1等于2"},
{"input": "法国的首都是？", "output": "巴黎"}
]
# 2.定义示例的消息格式提示词模版
msg_example_prompt = ChatPromptTemplate.from_messages([
("human", "{input}"),
("ai", "{output}"),
])
# 3.定义FewShotChatMessagePromptTemplate对象
few_shot_prompt = FewShotChatMessagePromptTemplate(
example_prompt=msg_example_prompt,
examples=examples
)
# 4.输出格式化后的消息
print(few_shot_prompt.format())

Human: 1+1等于几？
AI: 1+1等于2
Human: 法国的首都是？
AI: 巴黎


In [44]:
# 1.导入相关包
from langchain_core.prompts import (FewShotChatMessagePromptTemplate,
ChatPromptTemplate)
# 2.定义示例组
examples = [
{"input": "2🦜2", "output": "4"},
{"input": "2🦜3", "output": "8"},
]
# 3.定义示例的消息格式提示词模版
example_prompt = ChatPromptTemplate.from_messages([
('human', '{input} 是多少?'),
('ai', '{output}')
])
# 4.定义FewShotChatMessagePromptTemplate对象
few_shot_prompt = FewShotChatMessagePromptTemplate(
examples=examples, # 示例组
example_prompt=example_prompt, # 示例提示词词模版
)
# 5.输出完整提示词的消息模版
final_prompt = ChatPromptTemplate.from_messages(
[
('system', '你是一个数学奇才'),
few_shot_prompt,
('human', '{input}'),
]
)
#6.提供大模型
from langchain_ollama import ChatOllama
chat_model = ChatOllama(model="qwen:7b", temperature=0.4)
chat_model.invoke(final_prompt.invoke(input="2🦜4")).content

'16'

In [45]:
# 1.导入相关包
from langchain_core.prompts import (FewShotChatMessagePromptTemplate,
ChatPromptTemplate)
# 2.定义示例组
examples = [
{"input": "2+2", "output": "4"},
{"input": "2+3", "output": "5"},
]
# 3.定义示例的消息格式提示词模版
example_prompt = ChatPromptTemplate.from_messages([('human', 'What is {input}?'), ('ai', '{output}')])
# 4.定义FewShotChatMessagePromptTemplate对象
few_shot_prompt = FewShotChatMessagePromptTemplate(
examples=examples, # 示例组
example_prompt=example_prompt, # 示例提示词词模版
)
# 5.输出完整提示词的消息模版
final_prompt = ChatPromptTemplate.from_messages(
[
('system', 'You are a helpful AI Assistant'),
few_shot_prompt,
('human', '{input}'),
]
)
# 6.格式化完整消息
#final_prompt.format(input="What is 4+4?")
# 或者
final_prompt.format_messages(input="What is 4+4?")

[SystemMessage(content='You are a helpful AI Assistant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is 2+3?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='5', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is 4+4?', additional_kwargs={}, response_metadata={})]

#### 4.5.4 Example selectors(示例选择器)
前面FewShotPromptTemplate的特点是，无论输入什么问题，都会包含全部示例。在实际开发中，我
们可以根据当前输入，使用示例选择器，从大量候选示例中选取最相关的示例子集。
使用的好处：避免盲目传递所有示例，减少 token 消耗的同时，还可以提升输出效果。

示例选择策略：语义相似选择、长度选择、最大边际相关示例选择等

语义相似选择 ：通过余弦相似度等度量方式评估语义相关性，选择与输入问题最相似的 k 个示
例。

长度选择 ：根据输入文本的长度，从候选示例中筛选出长度最匹配的示例。增强模型对文本结构的
理解。比语义相似度计算更轻量，适合对响应速度要求高的场景。

最大边际相关示例选择 ：优先选择与输入问题语义相似的示例；同时，通过惩罚机制避免返回同质
化的内容

In [63]:
# 1.导入相关包
import os
from langchain_community.vectorstores import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import OllamaEmbeddings

# 2.定义嵌入模型
embeddings_model = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)
# embeddings_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# 3.定义示例组
examples = [
{
"question": "谁活得更久，穆罕默德·阿里还是艾伦·图灵?",
"answer": """
接下来还需要问什么问题吗？
追问：穆罕默德·阿里去世时多大年纪？
中间答案：穆罕默德·阿里去世时享年74岁。
""",
},
{
"question": "craigslist的创始人是什么时候出生的？",
"answer": """
接下来还需要问什么问题吗？
追问：谁是craigslist的创始人？
中级答案：Craigslist是由克雷格·纽马克创立的。
""",
},
{
"question": "谁是乔治·华盛顿的外祖父？",
"answer": """
接下来还需要问什么问题吗？
追问：谁是乔治·华盛顿的母亲？
中间答案：乔治·华盛顿的母亲是玛丽·鲍尔·华盛顿。
""",
},
{
"question": "《大白鲨》和《皇家赌场》的导演都来自同一个国家吗？",
"answer": """
接下来还需要问什么问题吗？
追问：《大白鲨》的导演是谁？
中级答案：《大白鲨》的导演是史蒂文·斯皮尔伯格。
""",
},
]
# 4.定义示例选择器
example_selector = SemanticSimilarityExampleSelector.from_examples(
# 这是可供选择的示例列表
examples,
# 这是用于生成嵌入的嵌入类，用于衡量语义相似性
embeddings_model,
# 这是用于存储嵌入并进行相似性搜索的 VectorStore 类
Chroma,
# 这是要生成的示例数量
k=1,
)
# 选择与输入最相似的示例
question = "玛丽·鲍尔·华盛顿的父亲是谁?"
selected_examples = example_selector.select_examples({"question": question})
print(f"与输入最相似的示例：{selected_examples}") # nomic-embed-text效果并不好
# for example in selected_examples:
# print("\n")
# for k, v in example.items():
# print(f"{k}: {v}")

与输入最相似的示例：[{'answer': '\n接下来还需要问什么问题吗？\n追问：穆罕默德·阿里去世时多大年纪？\n中间答案：穆罕默德·阿里去世时享年74岁。\n', 'question': '谁活得更久，穆罕默德·阿里还是艾伦·图灵?'}]


In [64]:
# 1.导入相关包
from langchain_community.vectorstores import FAISS
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_ollama import OllamaEmbeddings
# 2.定义示例提示词模版
example_prompt = PromptTemplate.from_template(
template="Input: {input}\nOutput: {output}",
)
# 3.创建一个示例提示词模版
examples = [
{"input": "高兴", "output": "悲伤"},
{"input": "高", "output": "矮"},
{"input": "长", "output": "短"},
{"input": "精力充沛", "output": "无精打采"},
{"input": "阳光", "output": "阴暗"},
{"input": "粗糙", "output": "光滑"},
{"input": "干燥", "output": "潮湿"},
{"input": "富裕", "output": "贫穷"},
]
# 4.定义嵌入模型
embeddings = OllamaEmbeddings(
model="mxbai-embed-large"
)
# 5.创建语义相似性示例选择器
example_selector = SemanticSimilarityExampleSelector.from_examples(
examples,
embeddings,
FAISS,
k=2,
)
#或者
#example_selector = SemanticSimilarityExampleSelector(
# examples,
# embeddings,
# FAISS,
# k=2
#)
# 6.定义小样本提示词模版
similar_prompt = FewShotPromptTemplate(
example_selector=example_selector,
example_prompt=example_prompt,
prefix="给出每个词组的反义词",
suffix="Input: {word}\nOutput:",
input_variables=["word"],
)
response = similar_prompt.invoke({"word":"忧郁"})
print(response.text)

给出每个词组的反义词

Input: 长
Output: 短

Input: 干燥
Output: 潮湿

Input: 忧郁
Output:


## 5、Model I/O之Output Parsers
语言模型返回的内容通常都是字符串的格式（文本格式），但在实际AI应用开发过程中，往往希望model可以返回更直观、更格式化的内容，以确保应用能够顺利进行后续的逻辑处理。此时，LangChain提供的 输出解析器 就派上用场了。

输出解析器（Output Parser）负责获取 LLM 的输出并将其转换为更合适的格式。这在应用开发中及其重要。

### 5.1 输出解析器的分类
LangChain有许多不同类型的输出解析器

- StrOutputParser ：字符串解析器
- JsonOutputParser ：JSON解析器，确保输出符合特定JSON对象格式
- XMLOutputParser ：XML解析器，允许以流行的XML格式从LLM获取结果
- CommaSeparatedListOutputParser ：CSV解析器，模型的输出以逗号分隔，以列表形式返回输出
- DatetimeOutputParser ：日期时间解析器，可用于将 LLM 输出解析为日期时间格式

除了上述常用的输出解析器之外，还有：
- EnumOutputParser ：枚举解析器，将LLM的输出，解析为预定义的枚举值
- StructuredOutputParser ：将非结构化文本转换为预定义格式的结构化数据（如字典）
- OutputFixingParser ：输出修复解析器，用于自动修复格式错误的解析器，比如将返回的不符合
预期格式的输出，尝试修正为正确的结构化数据（如 JSON）
- RetryOutputParser ：重试解析器，当主解析器（如 JSONOutputParser）因格式错误无法解析
LLM 的输出时，通过调用另一个 LLM 自动修正错误，并重新尝试解析
### 5.2 具体解析器的使用
#### 5.2.1字符串解析器 StrOutputParser
StrOutputParser 简单地将 任何输入 转换为 字符串 。它是一个简单的解析器，从结果中提取content字段

举例：将一个对话模型的输出结果，解析为字符串输出

In [65]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama
chat_model = ChatOllama(model="qwen:7b")
messages = [
SystemMessage(content="将以下内容从英语翻译成中文"),
HumanMessage(content="It's a nice day today"),
]
result = chat_model.invoke(messages)
print(type(result))
print(result)
parser = StrOutputParser()
#使用parser处理model返回的结果
response = parser.invoke(result)
print(type(response))
print(response)

<class 'langchain_core.messages.ai.AIMessage'>
content='今天是个美好的日子。' additional_kwargs={} response_metadata={'model': 'qwen:7b', 'created_at': '2026-01-07T05:39:40.8936112Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9158606800, 'load_duration': 5223276500, 'prompt_eval_count': 27, 'prompt_eval_duration': 2815282500, 'eval_count': 6, 'eval_duration': 1106177800, 'logprobs': None, 'model_name': 'qwen:7b', 'model_provider': 'ollama'} id='lc_run--019b96f7-a914-7072-b0f0-b3be51b220c1-0' usage_metadata={'input_tokens': 27, 'output_tokens': 6, 'total_tokens': 33}
<class 'str'>
今天是个美好的日子。


#### 5.2.2 JSON解析器 JsonOutputParser
JsonOutputParser，即JSON输出解析器，是一种用于将大模型的 自由文本输出 转换为 结构化JSON数据 的工具。

适合场景：特别适用于需要严格结构化输出的场景，比如 API 调用、数据存储或下游任务处理。

实现方式

方式1：用户自己通过提示词指明返回Json格式

方式2：借助JsonOutputParser的 get_format_instructions() ，生成格式说明，指导模型输出
JSON 结构

In [67]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
chat_model = ChatOllama(model="qwen:7b")
chat_prompt_template = ChatPromptTemplate.from_messages([
("system","你是一个靠谱的{role}"),
("human","{question}")
])
parser = JsonOutputParser()
# 方式1：
# result = chat_model.invoke(chat_prompt_template.format_messages(role="人工智能专家",question="人工智能用英文怎么说？问题用q表示，答案用a表示，返回一个JSON格式"))
# print(result)
# print(type(result))
# parser.invoke(result)
# 方式2：
chain = chat_prompt_template | chat_model | parser
chain.invoke({"role":"人工智能专家","question" : "人工智能用英文怎么说？问题用q表示，答案用a表示，返回一个JSON格式"})

{'question': '人工智能', 'answer': 'Artificial Intelligence'}

In [68]:
from langchain_core.output_parsers import JsonOutputParser
output_parser = JsonOutputParser()
# 返回一些指令或模板，这些指令告诉系统如何解析或格式化输出数据
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

Return a JSON object.


In [74]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama

# 1. 创建聊天模型
chat_model = ChatOllama(model="qwen:7b")

# 2. 创建要求 JSON 输出的提示词模板
prompt = PromptTemplate.from_template(
    "请以JSON格式回答：{query}\n{format_instructions}"
)

# 3. 创建 JSON 解析器
parser = JsonOutputParser()

# 4. 获取格式说明
format_instructions = parser.get_format_instructions()

# 5. 创建链
chain = prompt | chat_model | parser

# 6. 执行链（需要确保模型按 JSON 格式输出）
output = chain.invoke({
    "query": "给我讲一个笑话",
    "format_instructions": format_instructions
})

print(output)

{'joke': '为什么袜子总是只丢一只？因为丢两只根本就不会发现呀！'}


#### 5.2.3 XML解析器 XMLOutputParser
XMLOutputParser，将模型的自由文本输出转换为可编程处理的 XML 数据。
如何实现：在 PromptTemplate 中指定 XML 格式要求，让模型返回 <tag>content</tag> 形式的数
据。

注意：XMLOutputParser 不会直接将模型的输出保持为原始XML字符串，而是会解析XML并转换成

Python字典 （或类似结构化的数据）。目的是为了方便程序后续处理数据，而不是单纯保留XML格
式。

In [75]:
# 初始化语言模型
chat_model = ChatOllama(model="qwen:7b")
# 测试模型的xml解析效果
actor_query = "生成汤姆·汉克斯的简短电影记录"
output = chat_model.invoke(f"""{actor_query}请将影片附在<movie></movie>标签中"""
)
print(type(output)) # <class 'langchain_core.messages.ai.AIMessage'>
print(output.content)  # 7b很不聪明

<class 'langchain_core.messages.ai.AIMessage'>
<movie>

标题：《阿甘正传》片段——汤姆·汉克斯饰演阿甘

导演：罗伯特·泽米吉斯

时间长度：约5分钟

剧情概述：
这段影片选取了电影《阿甘正传》的经典场景。汤姆·汉克斯饰演的阿甘，因为腿部残疾，他的生活充满了曲折。但无论何时，阿甘始终保持纯真的笑容和积极向上的态度。

影片附带：
视频链接：(请插入实际链接或提供在线观看服务)

影像截图：(提供几张精炼的截图)

注释：详细描述每个片段的内容

</movie>


In [76]:
from langchain_core.output_parsers import XMLOutputParser
output_parser = XMLOutputParser()
# 返回一些指令或模板，这些指令告诉系统如何解析或格式化输出数据
format_instructions = output_parser.get_format_instructions()
print(format_instructions)

The output should be formatted as a XML file.
1. Output should conform to the tags below.
2. If tags are not given, make them on your own.
3. Remember to always open and close all the tags.

As an example, for the tags ["foo", "bar", "baz"]:
1. String "<foo>
   <bar>
      <baz></baz>
   </bar>
</foo>" is a well-formatted instance of the schema.
2. String "<foo>
   <bar>
   </foo>" is a badly-formatted instance.
3. String "<foo>
   <tag>
   </tag>
</foo>" is a badly-formatted instance.

Here are the output tags:
```
None
```


In [77]:
# 1.导入相关包
from langchain_core.output_parsers import XMLOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
# 2. 初始化语言模型
chat_model = ChatOllama(model="qwen:7b")
# 3.测试模型的xml解析效果
actor_query = "生成汤姆·汉克斯的简短电影记录,使用中文回复"
# 4.定义XMLOutputParser对象
parser = XMLOutputParser()
# 5.定义提示词模版对象
# prompt = PromptTemplate(
# template="{query}\n{format_instructions}",
# input_variables=["query","format_instructions"],
# partial_variables={"format_instructions": parser.get_format_instructions()},
#)
prompt_template = PromptTemplate.from_template("{query}\n{format_instructions}")
prompt_template1 = prompt_template.partial(format_instructions=parser.get_format_instructions())
response = chat_model.invoke(prompt_template1.format(query=actor_query))
print(response.content)

```xml
<?xml version="1.0" encoding="UTF-8"?>
<tom_hanks_movies>
  <movie>
    <title>Tom Hanks' Early Film</title>
    <year>1988</year>
    <description>
      <p>This movie marked Tom Hanks' debut in Hollywood. In the film, he played a young and ambitious journalist who uncovers a dark secret.</p>
      <p>The movie received positive reviews for Hanks' performance, which showcased his versatility as an actor.</p>
    </description>
  </movie>
  <!-- Additional movies can be added here -->
</tom_hanks_movies>
```
In this example, I have provided two movies starring Tom Hanks in his early career. The XML output includes the necessary tags for movie titles, years, and descriptions.


In [78]:
# 方式1
response = chat_model.invoke(prompt_template1.format(query=actor_query))
result = parser.invoke(response)
print(result)
print(type(result))

# 方式2
# chain = prompt_template1 | chat_model | parser
# result = chain.invoke({"query":actor_query})
# print(result)
# print(type(result))

{'tom_hanks_movies': [{'movie': [{'title': "Tom Hanks' Early Role - Big"}, {'year': '1988'}, {'description': "\n            In this early role, Tom Hanks portrays a young boy named Jimmy who lives in an orphanage. The movie follows Jimmy's journey as he discovers his talent for playing the trumpet and参加一个音乐比赛.\n\n            Hanks' performance in Big is often cited as a turning point in his career. The movie was commercially successful and received positive reviews, solidifying Tom Hanks' status as a talented actor.\n        "}]}]}
<class 'dict'>


#### 5.2.4 列表解析器 CommaSeparatedListOutputParser
列表解析器：利用此解析器可以将模型的文本响应转换为一个用 逗号分隔的列表（List[str]） 。

In [79]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
output_parser = CommaSeparatedListOutputParser()
# 返回一些指令或模板，这些指令告诉系统如何解析或格式化输出数据
format_instructions = output_parser.get_format_instructions()
print(format_instructions)
messages = "大象,猩猩,狮子"
result = output_parser.parse(messages)
print(result)
print(type(result))

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
['大象', '猩猩', '狮子']
<class 'list'>


In [84]:
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import CommaSeparatedListOutputParser
# 初始化语言模型
chat_model = ChatOllama(model="qwen:7b")
# 创建解析器
output_parser = CommaSeparatedListOutputParser()
# 创建LangChain提示模板
chat_prompt = PromptTemplate.from_template(
"生成5个关于{text}的列表.\n\n{format_instructions}",
partial_variables={
"format_instructions": output_parser.get_format_instructions()
})
# 提示模板与输出解析器传递输出
# chat_prompt =
chat_prompt.partial(format_instructions=output_parser.get_format_instructions())
# 将提示和模型合并以进行调用
chain = chat_prompt | chat_model | output_parser
res = chain.invoke({"text": "电影"})
print(res)
print(type(res))

['1. "Inception"', 'The Dark Knight', 'Interstellar', '2. "Top 10 Best Hollywood Movies"', 'Greatest Cinematic Masterpieces', 'Must-Watch Film Collections', '3. "Sci-Fi Blockbusters"', 'Action-Adventure Masterpieces', 'Futuristic Films That Transcend Time', '4. "List of Oscar-winning Movies"', 'Academy Award Nominees in Cinema History', 'Oscar-Contender Movies for Collectors', '5. "Best Foreign Language Films"', 'International Cinema masterpieces translated into English', 'English Subtitle Movies that Span the Globe']
<class 'list'>


#### 日期解析器 DatetimeOutputParser (了解)
利用此解析器可以直接将LLM输出解析为日期时间格式。
get_format_instructions()： 获取日期解析的格式化指令，指令为：
"Write a datetime string
that matches the following pattern: '%Y-%m-%dT%H:%M:%S.%fZ'。
举例：1206-08-16T17:39:06.176399Z

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts.chat import HumanMessagePromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import DatetimeOutputParser
chat_model = ChatOllama(model="qwen:7b")
chat_prompt = ChatPromptTemplate.from_messages([
("system","{format_instructions}"),
("human", "{request}")
])
output_parser = DatetimeOutputParser()
chain = chat_prompt | chat_model | output_parser
resp = chain.invoke({"request":"中华人民共和国是什么时候成立的",
"format_instructions":output_parser.get_format_instructions()})
print(resp)
print(type(resp))